# 01 — Build Clean, Deduplicated, Stratified Split (v2)

**Why v2:** the first pass used perceptual-hash threshold 5, which over-merged distinct fundus images into phantom clusters of 289+ and inflated apparent leakage to 30.5%. A threshold sweep showed the real duplicates all sit at Hamming distance <= 2 and are 100% same-class. At threshold 2 the data forms 187 small duplicate clusters (largest 28), ~490 images (~14%) — a believable real-duplication rate.

**This notebook:** pHash every image; group by direct pairwise distance <= 2 via union-find; assign each group wholesale to one split (no near-duplicate straddles train/val/test); rebalance per class toward 70/15/15; verify zero leakage; save index files. Run once, then Save Version and attach to notebook 02.

In [1]:
# ============================================================
# CONFIG
# ============================================================
DATA_ROOT       = "/kaggle/input/datasets/shajinrp/diabetic-retinopathy/Dataset"
PHASH_THRESHOLD = 2        # direct pairwise Hamming distance <= this => duplicate
SPLIT_SEED      = 42
TRAIN_FRAC      = 0.70
VAL_FRAC        = 0.15     # test gets remaining 0.15
OUT_DIR         = "/kaggle/working"

import os
assert os.path.isdir(DATA_ROOT), f"DATA_ROOT not found: {DATA_ROOT}"
print("Dataset root OK")

Dataset root OK


In [2]:
!pip install ImageHash --quiet
import numpy as np, torch, imagehash
from itertools import combinations
from PIL import Image
from torchvision import datasets
from collections import defaultdict, Counter

folder = datasets.ImageFolder(root=DATA_ROOT)
paths  = [p for p, _ in folder.samples]
labels = [l for _, l in folder.samples]
class_names = folder.classes
N = len(paths)
print("Classes (index order):", class_names)
print("Total images:", N)

Classes (index order): ['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe']
Total images: 3554


In [3]:
# --- Hash every image ---
def phash(p):
    try:
        with Image.open(p) as im:
            return imagehash.phash(im.convert("RGB"))
    except Exception as e:
        print("hash fail", os.path.basename(p), e); return None

print("Hashing all images (slow cell)...")
hashes = [phash(p) for p in paths]
valid  = [i for i in range(N) if hashes[i] is not None]
print("Done. Hashed", len(valid), "images.")

Hashing all images (slow cell)...
Done. Hashed 3554 images.


In [4]:
# --- Group by DIRECT pairwise distance <= threshold (union-find) ---
# At threshold 2 chaining is negligible (verified: largest cluster = 28).
parent = list(range(N))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra

pairs = [(a, b) for a, b in combinations(valid, 2)
         if hashes[a] - hashes[b] <= PHASH_THRESHOLD]
for a, b in pairs: union(a, b)

groups = defaultdict(list)
for i in valid: groups[find(i)].append(i)
group_list = list(groups.values())
dup = [g for g in group_list if len(g) > 1]
print(f"{len(pairs)} direct duplicate pairs")
print(f"{N} images -> {len(group_list)} groups; {len(dup)} duplicate clusters; "
      f"largest {max((len(g) for g in dup), default=0)}")

mixed = [g for g in dup if len({labels[i] for i in g}) > 1]
print(f"MIXED-class duplicate clusters: {len(mixed)} (expected 0)")

403 direct duplicate pairs
3554 images -> 3251 groups; 187 duplicate clusters; largest 28
MIXED-class duplicate clusters: 0 (expected 0)


In [5]:
# --- Group-aware STRATIFIED split WITH rebalancing ---
# Per class, place each whole group into whichever split is furthest below its
# target, keeping groups intact while pulling ratios close to 70/15/15.
rng = np.random.default_rng(SPLIT_SEED)

def group_class(g):
    return Counter(labels[i] for i in g).most_common(1)[0][0]

by_class = defaultdict(list)
for g in group_list:
    by_class[group_class(g)].append(g)

train_idx, val_idx, test_idx = [], [], []
targets = {"train": TRAIN_FRAC, "val": VAL_FRAC, "test": 1 - TRAIN_FRAC - VAL_FRAC}

for cls, gs in by_class.items():
    n_imgs = sum(len(g) for g in gs)
    tgt = {k: v * n_imgs for k, v in targets.items()}
    cur = {"train": 0, "val": 0, "test": 0}
    buckets = {"train": train_idx, "val": val_idx, "test": test_idx}
    for g in sorted(gs, key=len, reverse=True):
        deficit = {k: (tgt[k] - cur[k]) / max(tgt[k], 1) for k in cur}
        pick = max(deficit, key=deficit.get)
        buckets[pick].extend(g)
        cur[pick] += len(g)

train_idx = sorted(train_idx); val_idx = sorted(val_idx); test_idx = sorted(test_idx)
print(f"Train {len(train_idx)}  Val {len(val_idx)}  Test {len(test_idx)}")
tot = len(train_idx) + len(val_idx) + len(test_idx)
print(f"Ratios: {len(train_idx)/tot:.2f} / {len(val_idx)/tot:.2f} / {len(test_idx)/tot:.2f}")

Train 2485  Val 533  Test 536
Ratios: 0.70 / 0.15 / 0.15


In [6]:
# --- VERIFY zero cross-split leakage + per-class table ---
tr_g = {find(i) for i in train_idx}
va_g = {find(i) for i in val_idx}
te_g = {find(i) for i in test_idx}
print("Train/Val  shared groups:", len(tr_g & va_g))
print("Train/Test shared groups:", len(tr_g & te_g))
print("Val/Test   shared groups:", len(va_g & te_g))
assert not ((tr_g & va_g) or (tr_g & te_g) or (va_g & te_g)), "LEAK: group spans splits!"
print("VERIFIED: no duplicate group spans splits.\n")

print(f"{'Class':<16}{'Train':>7}{'Val':>7}{'Test':>7}{'Tr%':>6}{'Va%':>6}{'Te%':>6}")
for c, name in enumerate(class_names):
    tr = sum(labels[i]==c for i in train_idx)
    va = sum(labels[i]==c for i in val_idx)
    te = sum(labels[i]==c for i in test_idx)
    t = tr+va+te
    print(f"{name:<16}{tr:>7}{va:>7}{te:>7}{100*tr/t:>6.0f}{100*va/t:>6.0f}{100*te/t:>6.0f}")

Train/Val  shared groups: 0
Train/Test shared groups: 0
Val/Test   shared groups: 0
VERIFIED: no duplicate group spans splits.

Class             Train    Val   Test   Tr%   Va%   Te%
Mild                369     79     79    70    15    15
Moderate            276     59     60    70    15    15
No_DR               677    145    146    70    15    15
Proliferate_DR      761    164    164    70    15    15
Severe              402     86     87    70    15    15


In [7]:
# --- SAVE single source of truth ---
np.save(f"{OUT_DIR}/clean_train_indices.npy", np.array(train_idx))
np.save(f"{OUT_DIR}/clean_val_indices.npy",   np.array(val_idx))
np.save(f"{OUT_DIR}/clean_test_indices.npy",  np.array(test_idx))
with open(f"{OUT_DIR}/class_names.txt", "w") as f:
    f.write("\n".join(class_names))
print("Saved clean_{train,val,test}_indices.npy + class_names.txt to", OUT_DIR)
print("Next: Save Version, then attach this output to notebook 02.")

Saved clean_{train,val,test}_indices.npy + class_names.txt to /kaggle/working
Next: Save Version, then attach this output to notebook 02.
